In [46]:
import pandas as pd
import joblib
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score

In [47]:
# Datensatz öffnen und Struktur lesen

df = pd.read_csv(r"Student_Performance (1).csv") # CSV öffnen

df.info() # Informationen bekommen

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25000 entries, 0 to 24999
Data columns (total 16 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   student_id             25000 non-null  int64  
 1   age                    25000 non-null  int64  
 2   gender                 25000 non-null  object 
 3   school_type            25000 non-null  object 
 4   parent_education       25000 non-null  object 
 5   study_hours            25000 non-null  float64
 6   attendance_percentage  25000 non-null  float64
 7   internet_access        25000 non-null  object 
 8   travel_time            25000 non-null  object 
 9   extra_activities       25000 non-null  object 
 10  study_method           25000 non-null  object 
 11  math_score             25000 non-null  float64
 12  science_score          25000 non-null  float64
 13  english_score          25000 non-null  float64
 14  overall_score          25000 non-null  float64
 15  fi

In [48]:
df.head(15) #Einblick erhalten

,student_id,age,gender,school_type,parent_education,study_hours,attendance_percentage,internet_access,travel_time,extra_activities,study_method,math_score,science_score,english_score,overall_score,final_grade
0,1,14,male,public,post graduate,3.1,84.3,yes,<15 min,yes,notes,42.7,55.4,57.0,53.1,e
1,2,18,female,public,graduate,3.7,87.8,yes,>60 min,no,textbook,57.6,68.8,64.8,61.3,d
2,3,17,female,private,post graduate,7.9,65.5,no,<15 min,no,notes,84.8,95.0,79.2,89.6,b
3,4,16,other,public,high school,1.1,58.1,no,15-30 min,no,notes,44.4,27.5,54.7,41.6,e
4,5,16,female,public,high school,1.3,61.0,yes,30-60 min,yes,group study,8.9,32.7,30.0,25.4,f
5,6,19,male,public,no formal,3.8,69.6,yes,>60 min,yes,coaching,51.5,78.3,63.9,63.5,d
6,7,14,female,private,post graduate,1.8,81.6,yes,30-60 min,no,textbook,41.9,29.4,39.2,39.1,f
7,8,18,female,private,post graduate,5.6,59.4,yes,>60 min,yes,group study,56.7,60.1,53.4,69.6,d
8,9,15,other,private,high school,3.2,89.6,yes,15-30 min,yes,mixed,54.1,59.5,38.3,55.2,d
9,10,14,female,public,diploma,6.8,62.4,yes,>60 min,no,mixed,71.9,70.4,81.3,69.6,d


In [49]:
# Regressionsmodell implementieren

# X (Features) definieren und y (Target) definieren
X = df[["study_hours", "attendance_percentage"]]
y = df["overall_score"]

# Daten aufteilen (80% trainieren und 20% testen)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=17)

# Modell definieren und trainieren
modell = LinearRegression()
modell.fit(X_train, y_train)

# Vorhersage definieren
y_prediction = modell.predict(X_test)

# Auswertung geben
# Wichtig: R^2 beschreibt wie genau das Modell ist, während der Mean Absolute Error (MAE) den durchschnittlichen Fehler angibt bzw., wie weit es vom eigentlichen Wert liegt.
print(f"R^2 (Genauigkeitsniveau: 1=Perfekt, 0=ähnlich, wie Durchschnitt, -1=Durchschnitt ist genauer): {r2_score(y_test, y_prediction)}")
print(f"Durchschnittsfehler (MAE): {mean_absolute_error(y_test, y_prediction)} Punkte")

R^2 (Genauigkeitsniveau: 1=Perfekt, 0=ähnlich, wie Durchschnitt, -1=Durchschnitt ist genauer): 0.9057942845742571
Durchschnittsfehler (MAE): 4.955555209033984 Punkte


In [50]:
# Modell präziser machen mit One-Hot-Encoding

# String-Werte mit 0 und 1 Kodieren
df_encoded = pd.get_dummies(df, columns=["parent_education", "school_type"], drop_first=True)

df_encoded.head()

,student_id,age,gender,study_hours,attendance_percentage,internet_access,travel_time,extra_activities,study_method,math_score,science_score,english_score,overall_score,final_grade,parent_education_graduate,parent_education_high school,parent_education_no formal,parent_education_phd,parent_education_post graduate,school_type_public
0,1,14,male,3.1,84.3,yes,<15 min,yes,notes,42.7,55.4,57.0,53.1,e,False,False,False,False,True,True
1,2,18,female,3.7,87.8,yes,>60 min,no,textbook,57.6,68.8,64.8,61.3,d,True,False,False,False,False,True
2,3,17,female,7.9,65.5,no,<15 min,no,notes,84.8,95.0,79.2,89.6,b,False,False,False,False,True,False
3,4,16,other,1.1,58.1,no,15-30 min,no,notes,44.4,27.5,54.7,41.6,e,False,True,False,False,False,True
4,5,16,female,1.3,61.0,yes,30-60 min,yes,group study,8.9,32.7,30.0,25.4,f,False,True,False,False,False,True


In [51]:
# Neues Regressionsmodell implementieren

# X (Features) und y (Zielwert) definieren
X_komplex = df_encoded[
    [
        "study_hours", 
        "attendance_percentage", 
        "parent_education_graduate", 
        "parent_education_high school", 
        "parent_education_no formal", 
        "parent_education_phd", 
        "parent_education_post graduate", 
        "school_type_public"
    ]
]
y_komplex = df_encoded["overall_score"]

# Daten für das Training aufteilen
X_komplex_train, X_komplex_test, y_komplex_train, y_komplex_test = train_test_split(X_komplex, y_komplex, test_size=0.2, random_state=17)

# Modell definieren und trainieren
modell_komplex = LinearRegression()
modell_komplex.fit(X_komplex_train, y_komplex_train)

# Werte vorhersagen
y_komplex_prediction = modell_komplex.predict(X_komplex_test)

# Werte ausgeben
print(f"R^2 (Genauigkeit): {r2_score(y_komplex_prediction, y_komplex_test)}")
print(f"Standard Fehler (MAE): {mean_absolute_error(y_komplex_prediction, y_komplex_test)}")

# Das neue Modell ist ungenauer als das alte, deswegen werde ich für die Konsole den alten Modell verwenden

R^2 (Genauigkeit): 0.8974254488640653
Standard Fehler (MAE): 4.954717227578388


In [52]:
# Einfache GUI zur Verwendung des einfacheren Modells

def main():
    print("-"*50)
    print("-"*17, "Notenvorhersage", "-"*16)
    print("-"*50)
    print("\n")

    # Features abfragen
    while True:
        try:
            study_hours = float(input("Bitte Lernstunden angeben (täglich): "))
            if study_hours > 10 or study_hours < 0:
                print("Bitte deine echte Lernzeit angeben.")
            else:
                break
        except ValueError:
            print("Bitte nur Zahlen angeben.")
    while True:
        try:
            attendance_percentage = float(input("Bitte Anwesenheit in % angeben: "))
            if attendance_percentage > 100 or attendance_percentage < 70:
                    print("Bitte deine echte Anwesehneit angeben.")
            else:
                break
        except ValueError:
            print("Bitte nur Zahlen angeben.")
                

    # Alle One-Hot-Encoding Features auf 0 setzen
    score_features = {column: 0 for column in X.columns}

    # Features im Dictionary definieren
    score_features["study_hours"] = study_hours
    score_features["attendance_percentage"] = attendance_percentage

    # Neuer Datensatz mit gewünschten Features erstellen
    new_df = pd.DataFrame([score_features])

    # Preis vorhersagen
    score_prediction = modell.predict(new_df)[0]

    # Wert ausgeben und Disclaimer anzeigen
    print("\n")
    print("-"*50)
    if score_prediction > 100:
        print(f"Deine vorgesehene Punktzahl wird 100 sein")
    elif score_prediction < 0:
        print(f"Deine vorgesehene Punktzahl wird 0 sein")
    else:
        print(f"Deine vorgesehene Punktzahl wird {score_prediction:.2f} sein")
    print("-"*50)
    print("\n")
    print("Bitte beachte, dass es sich um ein Modell handelt. Die Antworten könnten nicht zu 100% mit der endgültigen Note stimmen. Das Modell ist zu 90% genau und hat einen Standard Fehler von 4.95 Punkte.")

if __name__ == "__main__":
    main()

--------------------------------------------------
----------------- Notenvorhersage ----------------
--------------------------------------------------




Bitte Lernstunden angeben (täglich):  7
Bitte Anwesenheit in % angeben:  100




--------------------------------------------------
Deine vorgesehene Punktzahl wird 95.58 sein
--------------------------------------------------


Bitte beachte, dass es sich um ein Modell handelt. Die Antworten könnten nicht zu 100% mit der endgültigen Note stimmen. Das Modell ist zu 90% genau und hat einen Standard Fehler von 4.95 Punkte.


In [53]:
# Modell speichern
joblib.dump(modell, "linear_regression_model.pkl")

['linear_regression_model.pkl']